In [1]:
from deep_translator import GoogleTranslator
text = "Hello, how are you?"
target_lang = 'te'
trans = GoogleTranslator(source = 'en',target = target_lang).translate(text)
print(trans)

హలో, ఎలా ఉన్నారు?


In [2]:
import re
text = '   Hello \n hello   world    '
text =  re.sub(r'\s+',' ',text).strip()
print(text)

Hello hello world


# Basha — IndicTrans2 Translation Benchmark (pretrained, CPU)

**Goal:** measure how good the *pretrained* AI4Bharat **IndicTrans2** model is at
English -> Indian-language translation, on the FLORES evaluation set, **before** deciding
whether any fine-tuning is worth it.

### Honest design notes (read these)
- **Model:** `ai4bharat/indictrans2-en-indic-dist-200M` (the *distilled 200M* variant).
  The 1B model is far too slow on CPU; the 200M is built for this.
- **Cores:** we set `torch.set_num_threads(6)` and run **batched** inference. We do *not*
  spawn 6 processes each loading the model — that would use 6x the RAM and thrash the laptop.
  One model spread across 6 cores is the correct way to "use 6 cores".
- **Metric:** **chrF** (character-F score via `sacrebleu`) is the standard, fair metric for
  morphologically rich Indian languages. We also report BLEU. This is a *translation*
  benchmark, so we do NOT use CER (CER was for the TTS audio round-trip — a different thing).
- **Scope:** IndicTrans2 is **Indic-only**, so we benchmark the 6 Indian languages:
  Telugu, Tamil, Kannada, Malayalam, Marathi, Hindi.
- **References** come from the FLORES set itself (`samples/input/flores_evaluation_set.json`).


## 0. Install (Tier-2 — run once)

These are heavy packages, not in the default Tier-1 install. Uncomment and run once.
`torch` installs the CPU build automatically when there is no CUDA.

```
%pip install torch transformers sacrebleu pandas tabulate
%pip install git+https://github.com/VarunGumma/IndicTransToolkit.git
```
> If the git install is slow, `pip install IndicTransToolkit` also works on most setups.
> It provides `IndicProcessor`, which does the script normalization IndicTrans2 needs.


In [ ]:
# 1. Configuration + use 6 CPU cores
import os
# Tell the math libraries to use 6 threads BEFORE importing torch heavy ops.
os.environ["OMP_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"

import time, json
import torch
torch.set_num_threads(6)          # PyTorch intra-op parallelism across 6 cores

MODEL_NAME   = "ai4bharat/indictrans2-en-indic-dist-200M"
SAMPLE_SIZE  = 100                 # sentences per language
BATCH_SIZE   = 16                  # mini-batch for inference (memory-friendly)
NUM_BEAMS    = 5                   # lower to 1 if it's too slow on your CPU
EVAL_PATH    = "../samples/input/flores_evaluation_set.json"
REPORT_PATH  = "../samples/output/indictrans2_report.md"

# en column -> FLORES target code used by IndicTrans2
TARGETS = {
    "te": "tel_Telu",
    "ta": "tam_Taml",
    "kn": "kan_Knda",
    "ml": "mal_Mlym",
    "mr": "mar_Deva",
    "hi": "hin_Deva",
}
SRC_LANG = "eng_Latn"
print("Threads:", torch.get_num_threads(), "| device: CPU")


In [ ]:
# 2. Load + clean the FLORES evaluation data
with open(EVAL_PATH, encoding="utf-8") as f:
    data = json.load(f)

def clean(s):
    # Some rows have stray wrapping quotes; strip them + whitespace.
    return s.strip().strip('"').strip() if isinstance(s, str) else ""

rows = data[:SAMPLE_SIZE]
src_sentences = [clean(r["en"]) for r in rows]
references = {lang: [clean(r[lang]) for r in rows] for lang in TARGETS}
print("Loaded", len(src_sentences), "English source sentences.")
print("Example EN:", src_sentences[0][:90])


In [ ]:
# 3. Load the pretrained IndicTrans2 model + processor (~1-2 min on CPU)
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

try:
    from IndicTransToolkit.processor import IndicProcessor
except ImportError:
    from IndicTransToolkit import IndicProcessor   # older package layout

print("Loading tokenizer + model:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
model.eval()
ip = IndicProcessor(inference=True)
print("Model loaded.")


In [ ]:
# 4. Batched translation for one target language
def translate(sentences, tgt_code, batch_size=BATCH_SIZE, num_beams=NUM_BEAMS):
    out = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        # IndicTrans2 needs its own preprocessing (script tags / normalization).
        pre = ip.preprocess_batch(batch, src_lang=SRC_LANG, tgt_lang=tgt_code)
        inputs = tokenizer(pre, truncation=True, padding="longest",
                           return_tensors="pt", max_length=256)
        with torch.no_grad():
            gen = model.generate(**inputs, num_beams=num_beams,
                                 num_return_sequences=1, max_length=256)
        decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
        out.extend(ip.postprocess_batch(decoded, lang=tgt_code))
        print("   ", tgt_code, ":", min(i + batch_size, len(sentences)), "/", len(sentences), "done")
    return out


In [ ]:
# 5. Run the benchmark across all 6 Indian languages + score with chrF/BLEU
import sacrebleu, pandas as pd

results = []
predictions = {}
for lang, tgt_code in TARGETS.items():
    print("\nTranslating EN ->", lang, "(", tgt_code, ") ...")
    t0 = time.perf_counter()
    hyps = translate(src_sentences, tgt_code)
    elapsed = time.perf_counter() - t0
    predictions[lang] = hyps

    refs = references[lang]
    chrf = sacrebleu.corpus_chrf(hyps, [refs]).score
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    results.append({
        "Language": lang,
        "chrF (higher=better)": round(chrf, 2),
        "BLEU (higher=better)": round(bleu, 2),
        "Sentences": len(hyps),
        "Time (s)": round(elapsed, 1),
        "Sec/sentence": round(elapsed / max(len(hyps), 1), 2),
    })

df = pd.DataFrame(results)
df


In [ ]:
# 6. Eyeball a few translations vs the gold reference (sanity check)
lang = "te"
for i in range(3):
    print("EN  :", src_sentences[i][:90])
    print("PRED:", predictions[lang][i][:90])
    print("GOLD:", references[lang][i][:90])
    print("-" * 70)


In [ ]:
# 7. Save a Markdown report
os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("# IndicTrans2 Translation Benchmark (pretrained, distilled-200M, CPU)\n\n")
    f.write("- Model: `" + MODEL_NAME + "`\n")
    f.write("- Sentences per language: " + str(SAMPLE_SIZE) + " (FLORES)\n")
    f.write("- Metric: chrF / BLEU via sacrebleu (higher is better)\n")
    f.write("- Threads: " + str(torch.get_num_threads()) + " (CPU)\n\n")
    f.write(df.to_markdown(index=False))
    f.write("\n\n**Reading it:** chrF > ~50 is generally a strong translation. "
            "If pretrained scores are already high, fine-tuning on a small set is NOT worth it.\n")
print("Saved report ->", REPORT_PATH)


## 8. How to read this — and the fine-tuning decision

- **chrF** is the headline number. Rough guide: **chrF > 50 = strong**, 40-50 = decent,
  < 40 = weak. BLEU runs lower than chrF for Indian languages; treat it as secondary.
- **If pretrained scores are already high** (they usually are for IndicTrans2): **do not
  fine-tune.** Fine-tuning on ~1000 sentences overfits and tends to make general translation
  *worse*. The honest call is "pretrained is good enough — and I measured it."
- **If specific terms/domain phrasing are wrong**, that's a *targeted* LoRA fine-tune case
  (~20 min on a free Colab/Kaggle T4 GPU) — a "what I'd do next" talking point, not a CPU job.

> **Interview framing:** *"I benchmarked pretrained IndicTrans2 with chrF before training
> anything, found it already strong, and concluded fine-tuning wasn't justified for a small
> dataset — I'd only LoRA-tune for a specific domain gap."* Decision-with-evidence beats
> running a training job for its own sake.
